In [ ]:
%load_ext autoreload
%autoreload 2

import os
import torch
from scipy.stats import spearmanr
from matplotlib import pyplot as plt
from hydra import initialize_config_dir, compose
from hydra.utils import instantiate
from src.constants import BASEDIR
from transformers import PreTrainedTokenizerFast

os.chdir(BASEDIR)

In [ ]:
with initialize_config_dir(os.path.join(BASEDIR, "configs/model")):
    cfg = compose(config_name="gpt2tiny_seqpos")

In [ ]:
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file=os.path.join(BASEDIR, "src/data/components/profam_tokenizer.json"),
    unk_token="[UNK]",
    pad_token="[PAD]",
    bos_token="[start-of-document]",
    sep_token="[SEP]",
    mask_token="[MASK]",
    add_special_tokens=True,
)

In [ ]:
model = instantiate(cfg, tokenizer=tokenizer)

In [ ]:
model = model.__class__.load_from_checkpoint(os.path.join(BASEDIR, "logs/train/runs/2024-07-04_16-36-14/checkpoints/epoch_046.ckpt"))
model.eval()

In [ ]:
from src.data.proteingym import load_gym_msa_dataset, load_gym_dataset


dataset = load_gym_msa_dataset("BLAT_ECOLX_Jacquier_2013", tokenizer, num_proc=8)

In [ ]:
mutants_data = load_gym_dataset(
    ["BLAT_ECOLX_Jacquier_2013"],
    tokenizer,
    max_tokens=2048,
    use_seq_pos=True,
    keep_gaps=True,
)

In [ ]:
mutants_data["input_ids"].shape, mutants_data["completion_ids"].shape

In [ ]:
tokenizer.convert_ids_to_tokens(24), tokenizer.convert_ids_to_tokens(26)

In [ ]:
tokenizer.convert_tokens_to_ids("[SEP]")

In [ ]:
torch.argwhere(mutants_data["input_ids"][0] == 28)

In [ ]:
mutants_data["seq_pos"][0]

In [ ]:
mutants_data["completion_seq_pos"][0][0]

In [ ]:
mutants_data["seq_pos"].shape, mutants_data["input_ids"].shape

In [ ]:
mutants_data["input_ids"].shape[-1] + mutants_data["completion_ids"].shape[-1]

In [ ]:
mutants_data["completion_seq_pos"].shape, mutants_data["completion_ids"].shape

In [ ]:
with torch.no_grad():
    scores = model._score_seqs_kv_cache(
        mutants_data["input_ids"],
        mutants_data["completion_ids"],
        seq_pos=mutants_data["seq_pos"],
        completion_seq_pos=mutants_data["completion_seq_pos"],
        verbose=True,
    )

In [ ]:
scores.shape

In [ ]:
mutants_data["DMS_scores"].shape

In [ ]:
spearmanr(scores, mutants_data["DMS_scores"].view(-1))

In [ ]:
plt.scatter(scores, mutants_data["DMS_scores"])

In [ ]:
with torch.no_grad():
    scores = model._score_seqs_no_cache(
        mutants_data["input_ids"],
        mutants_data["completion_ids"],
        seq_pos=mutants_data["seq_pos"],
        completion_seq_pos=mutants_data["completion_seq_pos"],
        verbose=True,
    )